# 2. RMSNorm and RoPE

**Goal:** learn the two small operations that stabilise and position every
token vector.

- **RMSNorm** keeps activation scale under control.
- **RoPE** encodes *where* a token is by rotating its vector.

## Setup

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while not (REPO / "packages" / "deepseek_flash_v4_mini").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "packages"))
sys.path.insert(0, str(REPO))
import torch

torch.manual_seed(0)
print("repo root:", REPO.name)
print("torch:", torch.__version__)

repo root: deepseek_flash_lab
torch: 2.13.0+cu130


## 1. RMSNorm from the formula

```
RMS(x) = sqrt(mean(x^2) + eps)
y      = x / RMS(x) * gamma
```

There is no mean subtraction and no bias. `gamma` is learned and starts at 1.

In [2]:
from deepseek_flash_v4_mini.model.normalization import RMSNorm

x = torch.randn(3, 8) * 4.0
norm = RMSNorm(8)
y = norm(x)

manual = x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + norm.eps) * norm.weight
print("module vs manual, max abs diff:", (y - manual).abs().max().item())

rms_before = x.pow(2).mean(dim=-1).sqrt()
rms_after = y.pow(2).mean(dim=-1).sqrt()
print("RMS before:", [round(v, 3) for v in rms_before.tolist()])
print("RMS after :", [round(v, 3) for v in rms_after.tolist()])

module vs manual, max abs diff: 0.0
RMS before: [4.152, 2.539, 5.119]
RMS after : [1.0, 1.0, 1.0]


**Notice:** after normalization, the RMS of every row is about 1.

In [3]:
# Scale invariance: multiplying the input by 100 should barely change the output.
y_small = norm(x)
y_big = norm(x * 100.0)
print("max abs diff (x vs 100x):", (y_small - y_big).abs().max().item())

max abs diff (x vs 100x): 2.384185791015625e-07


## 2. RoPE: position as rotation

RoPE pairs up dimensions and rotates each pair by an angle proportional to
the token position. Key properties:

1. The vector length does not change.
2. `dot(q_i, k_j)` depends on the **relative** position `i - j`.

In [4]:
from deepseek_flash_v4_mini.model.rope import apply_rope, rope_freqs

d_head, T = 8, 6
q = torch.randn(1, 1, T, d_head)       # [B, H, T, d_head]
freqs = rope_freqs(d_head, T, q.device)  # [T, d_head/2] complex
q_rot = apply_rope(q, freqs=freqs)

print("q shape    :", tuple(q.shape))
print("freqs shape:", tuple(freqs.shape))
print("norm before:", [round(v, 4) for v in q.norm(dim=-1)[0, 0].tolist()])
print("norm after :", [round(v, 4) for v in q_rot.norm(dim=-1)[0, 0].tolist()])

q shape    : (1, 1, 6, 8)
freqs shape: (6, 4)
norm before: [1.6101, 2.9445, 2.0541, 2.5044, 4.2142, 2.1527]
norm after : [1.6101, 2.9445, 2.0541, 2.5044, 4.2142, 2.1527]


In [5]:
# Relative-position property: shift q and k together, dot stays the same.
torch.manual_seed(1)
q_base = torch.randn(1, 1, 1, d_head)
k_base = torch.randn(1, 1, 1, d_head)
freqs8 = rope_freqs(d_head, 8, q_base.device)

def dot_at(i, j):
    qi = apply_rope(q_base, freqs=freqs8[i:i + 1])
    kj = apply_rope(k_base, freqs=freqs8[j:j + 1])
    return float((qi * kj).sum())

for offset in (0, 1, 2):
    values = [dot_at(i, i - offset) for i in (2, 3, 4)]
    print(f"i-j={offset}:", [round(v, 4) for v in values])

i-j=0: [-1.2136, -1.2136, -1.2136]
i-j=1: [-0.8557, -0.8557, -0.8557]
i-j=2: [-0.0093, -0.0093, -0.0093]


**Notice:** for a fixed offset, all three values are equal. Changing the
absolute position does not matter; only the distance does.

**Why rotation?** Rotating is cheap, preserves length, and turns position into
a relative signal inside the attention dot product.

**Try it:** `docs/EXERCISES.md` exercise 2 changes the RoPE `base` and observes
which distances are affected.